## Entrega Final — Minería de Datos II

### Equipo
- Lucas Dugo
- Cristian Lo Giudice
- Rodolfo Berrone
- Santiago Ham Saguier
- Andrea Romeo

Pipeline: landing → bronze → silver → gold y al final subimos a Astra.
7 CSV en batch, JSONL en streaming, 5 tablas en `default_keyspace`.
En Colab: *Ejecutar todo*.

In [ ]:
!pip -q install pyspark astrapy

## Instalación
Levanto Spark en local y bajo el ruido de los logs.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as funciones
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType,
    DoubleType, DateType, TimestampType, IntegerType, LongType,
)
from pyspark.sql.window import Window

# arranco Spark en Colab
spark = (SparkSession.builder
         .appName("final-colab")
         .master("local[*]")  # uso todos los cores
         .config("spark.sql.shuffle.partitions", "4")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")  # menos spam en consola
print("Spark listo, version:", spark.version)

## Repo y datos
Clono el repo de GitHub y miro que estén los datos en `datalake/landing`.

In [ ]:
url_repo = "https://github.com/javi2481/mineria-ii-parcial-final.git"
carpeta_repo = "/content/mineria-ii-parcial-final"

if not os.path.exists(carpeta_repo):
    os.system(f"git clone {url_repo} {carpeta_repo}")
else:
    os.system(f"cd {carpeta_repo} && git pull --ff-only origin main")

base = os.path.join(carpeta_repo, "final")
ruta_landing = os.path.join(carpeta_repo, "datalake", "landing")

if not os.path.exists(ruta_landing):
    raise FileNotFoundError(f"No encuentro landing en: {ruta_landing}")

print("ok, landing en:", ruta_landing)
print("commit:", os.popen(f"cd {carpeta_repo} && git log -1 --oneline").read().strip())

## Rutas del datalake
Defino dónde guardo bronze, silver, gold y los checkpoints.

In [ ]:
ruta_datalake = os.path.join(base, "datalake")
ruta_bronze_csv = os.path.join(ruta_datalake, "bronze", "batch")
ruta_bronze_eventos = os.path.join(ruta_datalake, "bronze", "stream", "usage_events")
ruta_landing_eventos = os.path.join(ruta_landing, "usage_events_stream")
ruta_silver_limpio = os.path.join(ruta_datalake, "silver", "usage_events_clean")
ruta_silver_quarantine = os.path.join(ruta_datalake, "silver", "quarantine")
ruta_silver_tickets = os.path.join(ruta_datalake, "silver", "support_tickets_clean")
ruta_silver_billing = os.path.join(ruta_datalake, "silver", "billing_monthly_clean")
ruta_gold_usage = os.path.join(ruta_datalake, "gold", "org_daily_usage_by_service")
ruta_gold_revenue = os.path.join(ruta_datalake, "gold", "revenue_by_org_month")
ruta_gold_anomaly = os.path.join(ruta_datalake, "gold", "cost_anomaly_mart")
ruta_gold_tickets = os.path.join(ruta_datalake, "gold", "tickets_by_org_date")
ruta_gold_genai = os.path.join(ruta_datalake, "gold", "genai_tokens_by_org_date")
ruta_checkpoint_bronze = os.path.join(base, "checkpoints", "bronze_stream")
ruta_checkpoint_astra_usage = os.path.join(base, "checkpoints", "serving_usage")
ruta_checkpoint_astra_revenue = os.path.join(base, "checkpoints", "serving_revenue")
ruta_checkpoint_astra_anomaly = os.path.join(base, "checkpoints", "serving_anomaly")
ruta_checkpoint_astra_tickets = os.path.join(base, "checkpoints", "serving_tickets")
ruta_checkpoint_astra_genai = os.path.join(base, "checkpoints", "serving_genai")

## Bronze — clientes
Primer CSV: schema a mano y saco duplicados por `org_id`.

In [ ]:
ruta_csv = os.path.join(ruta_landing, "customers_orgs.csv")
ruta_salida = os.path.join(ruta_bronze_csv, "customers_orgs")

esquema = StructType([
    StructField("org_id", StringType()), StructField("org_name", StringType()),
    StructField("industry", StringType()), StructField("hq_region", StringType()),
    StructField("plan_tier", StringType()), StructField("is_enterprise", BooleanType()),
    StructField("signup_date", DateType()), StructField("sales_rep", StringType()),
    StructField("lifecycle_stage", StringType()), StructField("marketing_source", StringType()),
    StructField("nps_score", DoubleType()),
])

df = spark.read.option("header", True).schema(esquema).csv(ruta_csv)
df = df.withColumn("ingest_ts", funciones.current_timestamp())
df = df.withColumn("source_file", funciones.input_file_name())
df = df.dropDuplicates(["org_id"])

df.write.mode("overwrite").partitionBy("hq_region").parquet(ruta_salida)
print("customers_orgs:", df.count(), "filas")

## Bronze — usuarios
Mismo paso con `users.csv`.

In [ ]:
ruta_csv = os.path.join(ruta_landing, "users.csv")
ruta_salida = os.path.join(ruta_bronze_csv, "users")

esquema = StructType([
    StructField("user_id", StringType()), StructField("org_id", StringType()),
    StructField("email", StringType()), StructField("role", StringType()),
    StructField("active", BooleanType()), StructField("created_at", DateType()),
    StructField("last_login", DateType()),
])

df = spark.read.option("header", True).schema(esquema).csv(ruta_csv)
df = df.withColumn("ingest_ts", funciones.current_timestamp())
df = df.withColumn("source_file", funciones.input_file_name())
df = df.dropDuplicates(["user_id"])

df.write.mode("overwrite").parquet(ruta_salida)
print("users:", df.count(), "filas")

## Bronze — facturacion
`billing_monthly.csv` con dedupe por `invoice_id`.

In [ ]:
ruta_csv = os.path.join(ruta_landing, "billing_monthly.csv")
ruta_salida = os.path.join(ruta_bronze_csv, "billing_monthly")

esquema = StructType([
    StructField("invoice_id", StringType()), StructField("org_id", StringType()),
    StructField("month", DateType()), StructField("subtotal", DoubleType()),
    StructField("credits", DoubleType()), StructField("taxes", DoubleType()),
    StructField("currency", StringType()), StructField("exchange_rate_to_usd", DoubleType()),
])

df = spark.read.option("header", True).schema(esquema).csv(ruta_csv)
df = df.withColumn("ingest_ts", funciones.current_timestamp())
df = df.withColumn("source_file", funciones.input_file_name())
df = df.dropDuplicates(["invoice_id"])

df.write.mode("overwrite").partitionBy("month").parquet(ruta_salida)
print("billing_monthly:", df.count(), "filas")

## Bronze — recursos
`resources.csv` particionado por `service`.

In [ ]:
ruta_csv = os.path.join(ruta_landing, "resources.csv")
ruta_salida = os.path.join(ruta_bronze_csv, "resources")

esquema = StructType([
    StructField("resource_id", StringType()), StructField("org_id", StringType()),
    StructField("service", StringType()), StructField("region", StringType()),
    StructField("created_at", DateType()), StructField("state", StringType()),
    StructField("tags_json", StringType()),
])

df = spark.read.option("header", True).schema(esquema).csv(ruta_csv)
df = df.withColumn("ingest_ts", funciones.current_timestamp())
df = df.withColumn("source_file", funciones.input_file_name())
df = df.dropDuplicates(["resource_id"])

df.write.mode("overwrite").partitionBy("service").parquet(ruta_salida)
print("resources:", df.count(), "filas")

## Bronze — tickets
Leo `support_tickets.csv` para después armar el gold de tickets.

In [ ]:
ruta_csv = os.path.join(ruta_landing, "support_tickets.csv")
ruta_salida = os.path.join(ruta_bronze_csv, "support_tickets")

esquema = StructType([
    StructField("ticket_id", StringType()), StructField("org_id", StringType()),
    StructField("category", StringType()), StructField("severity", StringType()),
    StructField("created_at", TimestampType()), StructField("resolved_at", TimestampType()),
    StructField("csat", DoubleType()), StructField("sla_breached", BooleanType()),
])

df = spark.read.option("header", True).schema(esquema).csv(ruta_csv)
df = df.withColumn("ingest_ts", funciones.current_timestamp())
df = df.withColumn("source_file", funciones.input_file_name())
df = df.dropDuplicates(["ticket_id"])

df.write.mode("overwrite").partitionBy("severity").parquet(ruta_salida)
print("support_tickets:", df.count(), "filas")

## Bronze — marketing
`marketing_touches.csv` (maestro extra del dataset).

In [ ]:
ruta_csv = os.path.join(ruta_landing, "marketing_touches.csv")
ruta_salida = os.path.join(ruta_bronze_csv, "marketing_touches")

esquema = StructType([
    StructField("touch_id", StringType()), StructField("org_id", StringType()),
    StructField("campaign", StringType()), StructField("channel", StringType()),
    StructField("timestamp", TimestampType()), StructField("clicked", BooleanType()),
    StructField("converted", BooleanType()),
])

df = spark.read.option("header", True).schema(esquema).csv(ruta_csv)
df = df.withColumn("ingest_ts", funciones.current_timestamp())
df = df.withColumn("source_file", funciones.input_file_name())
df = df.dropDuplicates(["touch_id"])

df.write.mode("overwrite").partitionBy("channel").parquet(ruta_salida)
print("marketing_touches:", df.count(), "filas")

## Bronze — nps
`nps_surveys.csv`.

In [ ]:
ruta_csv = os.path.join(ruta_landing, "nps_surveys.csv")
ruta_salida = os.path.join(ruta_bronze_csv, "nps_surveys")

esquema = StructType([
    StructField("org_id", StringType()), StructField("survey_date", DateType()),
    StructField("nps_score", DoubleType()), StructField("comment", StringType()),
])

df = spark.read.option("header", True).schema(esquema).csv(ruta_csv)
df = df.withColumn("ingest_ts", funciones.current_timestamp())
df = df.withColumn("source_file", funciones.input_file_name())
df = df.dropDuplicates(["org_id", "survey_date"])

df.write.mode("overwrite").parquet(ruta_salida)
print("nps_surveys:", df.count(), "filas")

## Esquema de eventos
Defino las columnas del JSONL antes del stream.

In [ ]:
esquema_eventos = StructType([
    StructField("event_id", StringType()),
    StructField("timestamp", TimestampType()),
    StructField("org_id", StringType()),
    StructField("resource_id", StringType()),
    StructField("service", StringType()),
    StructField("region", StringType()),
    StructField("metric", StringType()),
    StructField("value", StringType()),
    StructField("unit", StringType()),
    StructField("cost_usd_increment", DoubleType()),
    StructField("schema_version", IntegerType()),
    StructField("carbon_kg", DoubleType()),
    StructField("genai_tokens", LongType()),
])

## Bronze streaming
Leo JSONL en streaming, saco duplicados y guardo en bronze.

In [ ]:
!rm -rf "{ruta_bronze_eventos}" "{ruta_checkpoint_bronze}"
os.makedirs(ruta_bronze_eventos, exist_ok=True)

stream_entrada = (spark.readStream
                  .schema(esquema_eventos)
                  .option("maxFilesPerTrigger", 10)
                  .json(ruta_landing_eventos))

bronze_eventos = stream_entrada.withColumn("ingest_ts", funciones.current_timestamp())
bronze_eventos = bronze_eventos.withColumn("source_file", funciones.input_file_name())
bronze_eventos = bronze_eventos.withWatermark("timestamp", "2 days")
bronze_eventos = bronze_eventos.dropDuplicates(["event_id"])

query_bronze = (bronze_eventos.writeStream
                .format("parquet")
                .option("path", ruta_bronze_eventos)
                .option("checkpointLocation", ruta_checkpoint_bronze)
                .outputMode("append")
                .trigger(availableNow=True)
                .start())

query_bronze.awaitTermination()

## Conteo en bronze
Cuento filas totales y cuántos `event_id` distintos hay.

In [ ]:
eventos_bronze = spark.read.parquet(ruta_bronze_eventos)
total = eventos_bronze.count()
unicos = eventos_bronze.select("event_id").distinct().count()
print("eventos en bronze:", total, "| ids unicos:", unicos)

## Silver — reglas de calidad
Aplico R1, R2 y R3. Marco spikes con p99×3.

In [ ]:
eventos = spark.read.parquet(ruta_bronze_eventos)

eventos = eventos.withColumn("value_num", funciones.col("value").cast("double"))
eventos = eventos.withColumn("usage_date", funciones.to_date("timestamp"))
eventos = eventos.withColumn("schema_version", funciones.coalesce(funciones.col("schema_version"), funciones.lit(1)))
eventos = eventos.withColumn("carbon_kg", funciones.coalesce(funciones.col("carbon_kg"), funciones.lit(0.0)))
eventos = eventos.withColumn("genai_tokens", funciones.coalesce(funciones.col("genai_tokens"), funciones.lit(0)))

regla1 = funciones.col("event_id").isNull()
regla2 = funciones.col("cost_usd_increment") < -0.01
regla3 = funciones.col("value_num").isNotNull() & funciones.col("unit").isNull()

p99 = eventos.approxQuantile("cost_usd_increment", [0.99], 0.01)[0]
umbral_alto = p99 * 3
regla4_alto = funciones.col("cost_usd_increment") > umbral_alto

eventos = eventos.withColumn(
    "dq_rule",
    funciones.when(regla1, funciones.lit("R1_event_id_nulo"))
     .when(regla2, funciones.lit("R2_costo_anomalo"))
     .when(regla3, funciones.lit("R3_unit_nulo_con_value"))
     .otherwise(funciones.lit(None))
)
eventos = eventos.withColumn("cost_spike_flag", regla4_alto)

fila_invalida = regla1 | regla2 | regla3
df_limpio = eventos.filter(~fila_invalida)
df_quarantine = eventos.filter(fila_invalida)

df_quarantine.write.mode("overwrite").parquet(ruta_silver_quarantine)
print("quarantine:", df_quarantine.count(), "filas | p99 costo:", round(p99, 4), "| umbral:", round(umbral_alto, 4))
df_quarantine.select("event_id", "cost_usd_increment", "dq_rule").show(3, truncate=False)

## Silver — enriquecimiento
Join con clientes y recursos para sumar columnas al silver.

In [ ]:
clientes = (spark.read.parquet(os.path.join(ruta_bronze_csv, "customers_orgs"))
              .select("org_id", "org_name", "plan_tier"))
recursos = spark.read.parquet(os.path.join(ruta_bronze_csv, "resources")).select("resource_id", "org_id")

df_silver = df_limpio.join(clientes, on="org_id", how="left")
df_silver = df_silver.join(recursos, on=["resource_id", "org_id"], how="left")

df_silver = df_silver.withColumn(
    "cpu_hours",
    funciones.when(funciones.col("metric") == "cpu_hours", funciones.col("value_num")).otherwise(0.0)
)
df_silver = df_silver.withColumn(
    "storage_gb_hours",
    funciones.when(funciones.col("metric") == "storage_gb_hours", funciones.col("value_num")).otherwise(0.0)
)

columnas_silver = [
    "event_id", "timestamp", "usage_date", "org_id", "org_name", "plan_tier",
    "service", "region", "metric", "value_num", "unit",
    "cost_usd_increment", "schema_version", "carbon_kg", "genai_tokens",
    "cpu_hours", "storage_gb_hours", "cost_spike_flag",
]

df_silver.select(*columnas_silver).write.mode("overwrite").partitionBy("usage_date").parquet(ruta_silver_limpio)
print("silver limpio:", df_silver.count(), "filas")

## Silver — tickets y billing
Dejo listos tickets y facturación para armar el gold.

In [ ]:
tickets = spark.read.parquet(os.path.join(ruta_bronze_csv, "support_tickets"))
tickets = tickets.withColumn("ticket_date", funciones.to_date("created_at"))
tickets = tickets.withColumn(
    "is_critical", funciones.col("severity").isin("critical", "high")
)
tickets.write.mode("overwrite").partitionBy("ticket_date").parquet(ruta_silver_tickets)

billing = spark.read.parquet(os.path.join(ruta_bronze_csv, "billing_monthly"))
billing = billing.withColumn(
    "revenue_usd",
    (funciones.col("subtotal") - funciones.coalesce(funciones.col("credits"), funciones.lit(0.0))
     + funciones.coalesce(funciones.col("taxes"), funciones.lit(0.0))) * funciones.col("exchange_rate_to_usd")
)
billing = billing.join(clientes, on="org_id", how="left")
billing.write.mode("overwrite").partitionBy("month").parquet(ruta_silver_billing)
print("tickets silver:", tickets.count(), "| billing silver:", billing.count())

## Gold — uso diario (org_daily_usage_by_service)
Agrupo uso por org, fecha y servicio.

In [ ]:
df_silver = spark.read.parquet(ruta_silver_limpio)

gold_usage = df_silver.groupBy("org_id", "org_name", "plan_tier", "usage_date", "service").agg(
    funciones.round(funciones.sum("cost_usd_increment"), 4).alias("total_cost_usd"),
    funciones.sum(funciones.when(funciones.col("metric") == "requests", funciones.col("value_num")).otherwise(0))
     .cast("long").alias("total_requests"),
    funciones.round(funciones.sum("cpu_hours"), 4).alias("total_cpu_hours"),
    funciones.round(funciones.sum("storage_gb_hours"), 4).alias("total_storage_gb_hours"),
    funciones.sum(funciones.col("genai_tokens")).cast("long").alias("total_genai_tokens"),
    funciones.round(funciones.sum("carbon_kg"), 6).alias("total_carbon_kg"),
    funciones.count("*").cast("int").alias("event_count"),
)
gold_usage.write.mode("overwrite").partitionBy("usage_date").parquet(ruta_gold_usage)
print("gold usage:", gold_usage.count(), "filas")
gold_usage.show(3, truncate=False)

## Gold — facturación mensual (revenue_by_org_month)
Cuánto facturó cada org por mes.

In [ ]:
billing = spark.read.parquet(ruta_silver_billing)
gold_revenue = billing.groupBy("org_id", "org_name", "plan_tier", "month").agg(
    funciones.round(funciones.sum("revenue_usd"), 2).alias("revenue_usd"),
    funciones.round(funciones.sum(funciones.coalesce(funciones.col("credits"), funciones.lit(0.0))), 2).alias("total_credits"),
    funciones.round(funciones.sum(funciones.coalesce(funciones.col("taxes"), funciones.lit(0.0))), 2).alias("total_taxes"),
    funciones.first("currency").alias("currency"),
)
gold_revenue.write.mode("overwrite").partitionBy("month").parquet(ruta_gold_revenue)
print("gold revenue:", gold_revenue.count(), "filas")
gold_revenue.show(3, truncate=False)

## Gold — anomalías de costo (cost_anomaly_mart)
Solo los picos que pasan p99×3 (puede quedar vacío).

In [ ]:
df_silver = spark.read.parquet(ruta_silver_limpio)
gold_anomaly = (
    df_silver.filter(funciones.col("cost_spike_flag"))
    .groupBy("org_id", "org_name", "usage_date", "service")
    .agg(
        funciones.count("*").cast("int").alias("spike_event_count"),
        funciones.round(funciones.max("cost_usd_increment"), 4).alias("max_cost_usd"),
        funciones.round(funciones.avg("cost_usd_increment"), 4).alias("avg_cost_usd"),
        funciones.lit("p99_x3").alias("anomaly_method"),
        funciones.lit(True).alias("is_anomaly"),
    )
)
gold_anomaly.write.mode("overwrite").partitionBy("usage_date").parquet(ruta_gold_anomaly)
filas_anomaly = gold_anomaly.count()
print("gold anomaly:", filas_anomaly, "filas")
if filas_anomaly == 0:
    print("(sin spikes p99×3: no hay parquet, se omite carga a Astra)")
gold_anomaly.show(3, truncate=False)

## Gold — tickets por día (tickets_by_org_date)
Cuento tickets por org, día y severidad.

In [ ]:
tickets = spark.read.parquet(ruta_silver_tickets)
gold_tickets = tickets.groupBy("org_id", "ticket_date", "severity").agg(
    funciones.count("*").cast("int").alias("ticket_count"),
    funciones.sum(funciones.when(funciones.col("is_critical"), 1).otherwise(0)).cast("int").alias("critical_count"),
    funciones.round(
        funciones.avg(funciones.when(funciones.col("sla_breached"), 1.0).otherwise(0.0)), 4
    ).alias("sla_breach_rate"),
    funciones.round(funciones.avg("csat"), 2).alias("avg_csat"),
)
gold_tickets.write.mode("overwrite").partitionBy("ticket_date").parquet(ruta_gold_tickets)
print("gold tickets:", gold_tickets.count(), "filas")
gold_tickets.show(3, truncate=False)

## Gold — tokens GenAI (genai_tokens_by_org_date)
Suma tokens del servicio genai por org y día.

In [ ]:
df_silver = spark.read.parquet(ruta_silver_limpio)
genai = df_silver.filter(funciones.col("service") == "genai")
gold_genai = genai.groupBy("org_id", "org_name", "usage_date").agg(
    funciones.sum("genai_tokens").cast("long").alias("total_tokens"),
    funciones.round(funciones.sum("cost_usd_increment"), 4).alias("estimated_cost_usd"),
)
gold_genai.write.mode("overwrite").partitionBy("usage_date").parquet(ruta_gold_genai)
print("gold genai:", gold_genai.count(), "filas")
gold_genai.show(3, truncate=False)

## Token Astra
Saco el token del Secret `ASTRA_TOKEN_FINAL` en Colab (cluster `db_final_istea`).

In [ ]:
try:
    from google.colab import userdata
    token_astra = userdata.get("ASTRA_TOKEN_FINAL")
except Exception:
    token_astra = os.environ.get("ASTRA_TOKEN_FINAL", "")

endpoint_astra = "https://c14f9098-f0de-4d1c-8b7a-667a57943f2c-us-east-2.apps.astra.datastax.com"
keyspace = "default_keyspace"

if not token_astra:
    raise ValueError("Falta ASTRA_TOKEN_FINAL: Secret en Colab o variable de entorno")

## Tablas en Astra
Creo las 5 tablas en `default_keyspace`.

In [ ]:
from astrapy import DataAPIClient
from astrapy.info import CreateTableDefinition, ColumnType

cliente = DataAPIClient()
bd = cliente.get_database(endpoint_astra, token=token_astra)

def crear_tabla(nombre, columnas, partition_by, sort_map=None):
    builder = CreateTableDefinition.builder()
    for col, tipo in columnas:
        builder = builder.add_column(col, tipo)
    builder = builder.add_partition_by(partition_by)
    if sort_map:
        builder = builder.add_partition_sort(sort_map)
    definicion = builder.build()
    if nombre in bd.list_collection_names(keyspace=keyspace):
        bd.drop_collection(nombre, keyspace=keyspace)
    try:
        bd.drop_table(nombre, keyspace=keyspace)
    except Exception:
        pass
    return bd.create_table(nombre, definition=definicion, keyspace=keyspace)

tabla_usage = crear_tabla(
    "org_daily_usage_by_service",
    [
        ("org_id", ColumnType.TEXT), ("usage_date", ColumnType.DATE), ("service", ColumnType.TEXT),
        ("org_name", ColumnType.TEXT), ("plan_tier", ColumnType.TEXT),
        ("total_cost_usd", ColumnType.DOUBLE), ("total_requests", ColumnType.BIGINT),
        ("total_cpu_hours", ColumnType.DOUBLE), ("total_storage_gb_hours", ColumnType.DOUBLE),
        ("total_genai_tokens", ColumnType.BIGINT), ("total_carbon_kg", ColumnType.DOUBLE),
        ("event_count", ColumnType.INT),
    ],
    ["org_id", "usage_date"], {"service": 1},
)

tabla_revenue = crear_tabla(
    "revenue_by_org_month",
    [
        ("org_id", ColumnType.TEXT), ("month", ColumnType.DATE),
        ("org_name", ColumnType.TEXT), ("plan_tier", ColumnType.TEXT),
        ("revenue_usd", ColumnType.DOUBLE), ("total_credits", ColumnType.DOUBLE),
        ("total_taxes", ColumnType.DOUBLE), ("currency", ColumnType.TEXT),
    ],
    ["org_id"], {"month": 1},
)

tabla_anomaly = crear_tabla(
    "cost_anomaly_mart",
    [
        ("org_id", ColumnType.TEXT), ("usage_date", ColumnType.DATE), ("service", ColumnType.TEXT),
        ("org_name", ColumnType.TEXT), ("spike_event_count", ColumnType.INT),
        ("max_cost_usd", ColumnType.DOUBLE), ("avg_cost_usd", ColumnType.DOUBLE),
        ("anomaly_method", ColumnType.TEXT), ("is_anomaly", ColumnType.BOOLEAN),
    ],
    ["org_id", "usage_date"], {"service": 1},
)

tabla_tickets = crear_tabla(
    "tickets_by_org_date",
    [
        ("org_id", ColumnType.TEXT), ("ticket_date", ColumnType.DATE), ("severity", ColumnType.TEXT),
        ("ticket_count", ColumnType.INT), ("critical_count", ColumnType.INT),
        ("sla_breach_rate", ColumnType.DOUBLE), ("avg_csat", ColumnType.DOUBLE),
    ],
    ["org_id"], {"ticket_date": 1, "severity": 1},
)

tabla_genai = crear_tabla(
    "genai_tokens_by_org_date",
    [
        ("org_id", ColumnType.TEXT), ("usage_date", ColumnType.DATE),
        ("org_name", ColumnType.TEXT), ("total_tokens", ColumnType.BIGINT),
        ("estimated_cost_usd", ColumnType.DOUBLE),
    ],
    ["org_id"], {"usage_date": 1},
)

print("tablas listas en", keyspace)

## Carga a Astra
Subo cada tabla gold a Astra. Inserto de a pocos registros porque si no hace timeout.

In [ ]:
from astrapy.data_types import DataAPIDate
import shutil
import time
from astrapy.exceptions.data_api_exceptions import DataAPITimeoutException

def insertar_con_reintento(tabla, filas, tam_lote=10, max_intentos=3):
    # mando de a 10 filas con reintento (evita timeout y cortes de red)
    for i in range(0, len(filas), tam_lote):
        lote = filas[i:i + tam_lote]
        for intento in range(max_intentos):
            try:
                tabla.insert_many(lote, chunk_size=tam_lote, ordered=True, concurrency=1)
                break
            except DataAPITimeoutException:
                if intento == max_intentos - 1:
                    raise
                time.sleep(5)

def cargar_parquet_a_tabla(nombre, ruta_gold, tabla, mapeo_fn, checkpoint):
    def tiene_parquet(carpeta):
        for _, _, archivos in os.walk(carpeta):
            if any(a.endswith(".parquet") for a in archivos):
                return True
        return False

    if not os.path.exists(ruta_gold) or not tiene_parquet(ruta_gold):
        print(f"{nombre}: gold vacío (sin .parquet), omito carga Astra")
        return

    if os.path.exists(checkpoint):
        shutil.rmtree(checkpoint)
    os.makedirs(os.path.dirname(checkpoint), exist_ok=True)

    def cargar(batch_df, batch_id):
        filas = [mapeo_fn(fila) for fila in batch_df.collect()]
        if filas:
            insertar_con_reintento(tabla, filas)
        print(f"{nombre} batch {batch_id}: {len(filas)} filas")

    (spark.readStream
     .schema(spark.read.parquet(ruta_gold).schema)
     .parquet(ruta_gold)
     .writeStream
     .foreachBatch(cargar)
     .option("checkpointLocation", checkpoint)
     .trigger(availableNow=True)
     .start()
     .awaitTermination())

cargar_parquet_a_tabla(
    "org_daily_usage_by_service", ruta_gold_usage, tabla_usage,
    lambda f: {
        "org_id": f["org_id"],
        "usage_date": DataAPIDate.from_string(str(f["usage_date"])),
        "service": f["service"],
        "org_name": f["org_name"],
        "plan_tier": f["plan_tier"],
        "total_cost_usd": float(f["total_cost_usd"] or 0),
        "total_requests": int(f["total_requests"] or 0),
        "total_cpu_hours": float(f["total_cpu_hours"] or 0),
        "total_storage_gb_hours": float(f["total_storage_gb_hours"] or 0),
        "total_genai_tokens": int(f["total_genai_tokens"] or 0),
        "total_carbon_kg": float(f["total_carbon_kg"] or 0),
        "event_count": int(f["event_count"] or 0),
    },
    ruta_checkpoint_astra_usage,
)

cargar_parquet_a_tabla(
    "revenue_by_org_month", ruta_gold_revenue, tabla_revenue,
    lambda f: {
        "org_id": f["org_id"],
        "month": DataAPIDate.from_string(str(f["month"])),
        "org_name": f["org_name"],
        "plan_tier": f["plan_tier"],
        "revenue_usd": float(f["revenue_usd"] or 0),
        "total_credits": float(f["total_credits"] or 0),
        "total_taxes": float(f["total_taxes"] or 0),
        "currency": f["currency"],
    },
    ruta_checkpoint_astra_revenue,
)

cargar_parquet_a_tabla(
    "cost_anomaly_mart", ruta_gold_anomaly, tabla_anomaly,
    lambda f: {
        "org_id": f["org_id"],
        "usage_date": DataAPIDate.from_string(str(f["usage_date"])),
        "service": f["service"],
        "org_name": f["org_name"],
        "spike_event_count": int(f["spike_event_count"] or 0),
        "max_cost_usd": float(f["max_cost_usd"] or 0),
        "avg_cost_usd": float(f["avg_cost_usd"] or 0),
        "anomaly_method": f["anomaly_method"],
        "is_anomaly": bool(f["is_anomaly"]),
    },
    ruta_checkpoint_astra_anomaly,
)

cargar_parquet_a_tabla(
    "tickets_by_org_date", ruta_gold_tickets, tabla_tickets,
    lambda f: {
        "org_id": f["org_id"],
        "ticket_date": DataAPIDate.from_string(str(f["ticket_date"])),
        "severity": f["severity"],
        "ticket_count": int(f["ticket_count"] or 0),
        "critical_count": int(f["critical_count"] or 0),
        "sla_breach_rate": float(f["sla_breach_rate"] or 0),
        "avg_csat": float(f["avg_csat"]) if f["avg_csat"] is not None else None,
    },
    ruta_checkpoint_astra_tickets,
)

cargar_parquet_a_tabla(
    "genai_tokens_by_org_date", ruta_gold_genai, tabla_genai,
    lambda f: {
        "org_id": f["org_id"],
        "usage_date": DataAPIDate.from_string(str(f["usage_date"])),
        "org_name": f["org_name"],
        "total_tokens": int(f["total_tokens"] or 0),
        "estimated_cost_usd": float(f["estimated_cost_usd"] or 0),
    },
    ruta_checkpoint_astra_genai,
)

print("carga Astra completa")

## Consulta #1
Costos y requests de una org en un rango de fechas.

In [ ]:
import pandas as pd

org_consulta = "org_pbhsahxt"
fecha_desde = DataAPIDate.from_string("2025-07-01")
fecha_hasta = DataAPIDate.from_string("2025-08-31")

res1 = list(tabla_usage.find({"org_id": org_consulta}))
df1 = pd.DataFrame(res1)
if not df1.empty:
    df1 = df1[(df1["usage_date"] >= fecha_desde) & (df1["usage_date"] <= fecha_hasta)]
    df1 = df1.sort_values(["usage_date", "service"], key=lambda s: s.map(lambda x: x.to_string() if hasattr(x, "to_string") else str(x)))

print(f"consulta 1 — {org_consulta} jul-ago 2025:")
if not df1.empty:
    display(df1.assign(usage_date=df1["usage_date"].map(str))[
        ["usage_date", "service", "total_cost_usd", "total_requests", "event_count"]
    ])
else:
    print("(sin resultados)")

## Consulta #2
Top servicios por costo en los últimos 14 días.

In [ ]:
res2 = list(tabla_usage.find({"org_id": org_consulta}))
df2 = pd.DataFrame(res2)
fecha_corte = DataAPIDate.from_string("2025-08-18")
fecha_inicio = DataAPIDate.from_string("2025-08-05")

if not df2.empty:
    df2 = df2[(df2["usage_date"] >= fecha_inicio) & (df2["usage_date"] <= fecha_corte)]
    top = (
        df2.groupby("service", as_index=False)["total_cost_usd"].sum()
        .sort_values("total_cost_usd", ascending=False)
        .head(5)
    )
else:
    top = pd.DataFrame()

print(f"consulta 2 — top 5 servicios por costo, {org_consulta}, ~14 dias:")
if not top.empty:
    display(top)
else:
    print("(sin resultados)")

## Consulta #3
Tickets críticos/altos y SLA breach en los últimos 30 días.

In [ ]:
res3 = list(tabla_tickets.find({"org_id": org_consulta}))
df3 = pd.DataFrame(res3)
fecha_tickets_desde = DataAPIDate.from_string("2025-07-19")
fecha_tickets_hasta = DataAPIDate.from_string("2025-08-18")

if not df3.empty:
    df3 = df3[df3["severity"].isin(["critical", "high"])]
    df3 = df3[(df3["ticket_date"] >= fecha_tickets_desde) & (df3["ticket_date"] <= fecha_tickets_hasta)]
    df3 = df3.sort_values("ticket_date", key=lambda s: s.map(lambda d: d.to_string()))

print(f"consulta 3 — tickets criticos/altos (30 dias), {org_consulta}:")
if not df3.empty:
    display(df3.assign(ticket_date=df3["ticket_date"].map(str))[
        ["ticket_date", "severity", "ticket_count", "critical_count", "sla_breach_rate"]
    ])
else:
    print("(sin resultados)")

## Consulta #4
Revenue mensual en USD.

In [ ]:
res4 = list(tabla_revenue.find({"org_id": org_consulta}))
df4 = pd.DataFrame(res4)
if not df4.empty:
    df4 = df4.sort_values("month", key=lambda s: s.map(lambda d: d.to_string()))

print(f"consulta 4 — revenue mensual USD, {org_consulta}:")
if not df4.empty:
    display(df4.assign(month=df4["month"].map(str))[
        ["month", "revenue_usd", "total_credits", "total_taxes", "currency"]
    ])
else:
    print("(sin resultados)")

## Consulta #5
Tokens GenAI y costo estimado por día.

In [ ]:
res5 = list(tabla_genai.find({"org_id": org_consulta}))
df5 = pd.DataFrame(res5)
if not df5.empty:
    df5 = df5.sort_values("usage_date", key=lambda s: s.map(lambda d: d.to_string()))

print(f"consulta 5 — genai tokens por dia, {org_consulta}:")
if not df5.empty:
    display(df5.assign(usage_date=df5["usage_date"].map(str))[
        ["usage_date", "total_tokens", "estimated_cost_usd"]
    ])
else:
    print("(sin resultados)")

## Idempotencia del stream
Corro el stream otra vez: el conteo en bronze no debería cambiar.

In [ ]:
antes = spark.read.parquet(ruta_bronze_eventos).count()

stream_entrada2 = (spark.readStream
                   .schema(esquema_eventos)
                   .option("maxFilesPerTrigger", 10)
                   .json(ruta_landing_eventos))

bronze_eventos2 = stream_entrada2.withColumn("ingest_ts", funciones.current_timestamp())
bronze_eventos2 = bronze_eventos2.withColumn("source_file", funciones.input_file_name())
bronze_eventos2 = bronze_eventos2.withWatermark("timestamp", "2 days")
bronze_eventos2 = bronze_eventos2.dropDuplicates(["event_id"])

query_bronze2 = (bronze_eventos2.writeStream
                 .format("parquet")
                 .option("path", ruta_bronze_eventos)
                 .option("checkpointLocation", ruta_checkpoint_bronze)
                 .outputMode("append")
                 .trigger(availableNow=True)
                 .start())
query_bronze2.awaitTermination()

despues = spark.read.parquet(ruta_bronze_eventos).count()
print("bronze antes:", antes, "despues:", despues, "-> igual?", antes == despues)

## Tamaño en disco
Cuánto ocupa cada capa en MB.

In [ ]:
def tamano(carpeta):
    total = 0
    for ruta, _, archivos in os.walk(carpeta):
        for a in archivos:
            total += os.path.getsize(os.path.join(ruta, a))
    return round(total / 1024 / 1024, 2)

print("bronze batch:", tamano(ruta_bronze_csv), "MB")
print("bronze stream:", tamano(ruta_bronze_eventos), "MB")
print("silver:", tamano(os.path.join(ruta_datalake, "silver")), "MB")
print("gold:", tamano(os.path.join(ruta_datalake, "gold")), "MB")

## CQL de las 5 consultas

Las mismas queries están en `final/cql/02_queries.cql`. Capturas de resultados en `final/evidencias/consulta_1.png` … `consulta_5.png`.

```sql
-- #1 Costos y requests diarios
SELECT usage_date, service, total_cost_usd, total_requests, event_count
FROM default_keyspace.org_daily_usage_by_service
WHERE org_id = 'org_pbhsahxt'
  AND usage_date >= '2025-07-01' AND usage_date <= '2025-08-31';

-- #2 Top servicios por costo (~14 dias)
SELECT service, sum(total_cost_usd) AS costo_acumulado
FROM default_keyspace.org_daily_usage_by_service
WHERE org_id = 'org_pbhsahxt'
  AND usage_date >= '2025-08-05' AND usage_date <= '2025-08-18'
GROUP BY service;

-- #3 Tickets criticos/altos (ultimos 30 dias)
SELECT ticket_date, severity, ticket_count, critical_count, sla_breach_rate
FROM default_keyspace.tickets_by_org_date
WHERE org_id = 'org_pbhsahxt'
  AND severity IN ('critical', 'high')
  AND ticket_date >= '2025-07-19' AND ticket_date <= '2025-08-18';

-- #4 Revenue mensual USD
SELECT month, revenue_usd, total_credits, total_taxes, currency
FROM default_keyspace.revenue_by_org_month
WHERE org_id = 'org_pbhsahxt';

-- #5 Tokens GenAI por dia
SELECT usage_date, total_tokens, estimated_cost_usd
FROM default_keyspace.genai_tokens_by_org_date
WHERE org_id = 'org_pbhsahxt'
  AND usage_date >= '2025-07-01';
```

Arriba en este notebook las corremos con `astrapy.find` (misma org y filtros). Para la entrega también pegamos el CQL en la consola de Astra.